# 🦠 Pathogen Finder — Sistem AI de Identificare Agenți Patogeni
### Versiunea 2.0 — cu XGBoost, ROC AUC, SVM și Pydantic

---

## 🎯 Definirea Problemei

**Problema:** Un cercetător medical are nevoie să găsească rapid ce agenți patogeni sunt asociați cu o boală, pornind de la un catalog cu 700+ intrări. Căutarea manuală durează minute — sistemul nostru răspunde în sub 1 secundă.

**Contextul:** Catalogul `vega_pathogens_catalog.csv` conține agenți patogeni (bacterii, paraziți, virusuri) cu frecvențele lor, bolile asociate și organele afectate.

**Input:** Numele unei boli (ex: `ACNEE`, `sifilis`, `toxoplasmoza`)

**Output:** Lista agenților patogeni asociați, validată cu Pydantic, cu scoruri de relevanță

**De ce e important:** La fel cum la Titanic preziceam supraviețuitorii pe baza unor features (clasă, sex, vârstă), noi vom prezice relevanța unui agent patogen față de o boală căutată — tot o problemă de clasificare binară.

---

## 📚 Ce algoritmi folosim și de ce

| Algoritm | De ce îl folosim | Analogie curs |
|----------|-----------------|---------------|
| **TF-IDF + Cosine** | Baseline rapid, ca Naive Bayes | Clasificator simplu de referință |
| **XGBoost** | Pipeline de clasificatori care se corectează reciproc | Discutat în curs — câștigă competiții Kaggle |
| **SVM** | Găsește hyperplane-ul optim între clase | Support Vector Machine din curs |
| **Sentence Transformers** | Înțelege sensul semantic al textului | Modelul cel mai avansat |
| **ROC AUC** | Metrică robustă indiferent de threshold | Discutată în curs la Titanic Kaggle |

## Pasul 1 — Instalare librării

In [ ]:
!pip install pydantic sentence-transformers scikit-learn xgboost pandas matplotlib seaborn wordcloud numpy

## Pasul 2 — Importuri și creare foldere

In [ ]:
import os
os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import re
import pickle
import warnings
warnings.filterwarnings('ignore')

# Pydantic - validare date
from pydantic import BaseModel, Field
from typing import List, Optional

# Modele ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report
)
from xgboost import XGBClassifier
from sentence_transformers import SentenceTransformer

print('✅ Toate librăriile au fost importate!')
print('✅ Folderele au fost create!')

## Pasul 3 — Structuri Pydantic pentru validarea datelor

Pydantic ne asigură că datele returnate sunt întotdeauna complete și corecte — nu putem primi un rezultat cu câmpuri lipsă.

In [ ]:
class AgentPatogen(BaseModel):
    """
    Structura validată a unui agent patogen.
    Pydantic verifică automat toate câmpurile.
    """
    nume: str = Field(description="Numele agentului patogen")
    tip: str = Field(description="Tipul: Bacterie, Parazit, Necunoscut etc")
    frecventa: str = Field(description="Frecvența în Hz")
    boli_asociate: List[str] = Field(description="Lista bolilor asociate")
    organe_tinta: List[str] = Field(description="Lista organelor afectate")
    numar_boli: int = Field(ge=0, description="Numărul de boli asociate")
    scor_relevanta: float = Field(ge=0.0, le=1.0, description="Scor de relevanță 0-1")


class RezultatCautare(BaseModel):
    """
    Rezultatul complet al unei căutări.
    """
    boala_cautata: str
    agenti_gasiti: List[AgentPatogen]
    numar_rezultate: int
    model_folosit: str = Field(description="Ce model a găsit rezultatele")
    mesaj: str


print('✅ Structuri Pydantic definite!')
print()
print('AgentPatogen are câmpurile:')
for camp, info in AgentPatogen.model_fields.items():
    print(f'  → {camp}: {info.description}')

## Pasul 4 — Încărcare date CSV

In [ ]:
from google.colab import files
import io

# verificam daca fisierul exista deja
if os.path.exists('vega_pathogens_catalog.csv'):
    df = pd.read_csv('vega_pathogens_catalog.csv')
    print('✅ Fișierul găsit și încărcat automat!')
elif os.path.exists('data/vega_pathogens_catalog.csv'):
    df = pd.read_csv('data/vega_pathogens_catalog.csv')
    print('✅ Fișierul găsit în data/ și încărcat!')
else:
    print('📂 Te rog încarcă fișierul vega_pathogens_catalog.csv')
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(io.BytesIO(uploaded[filename]))
    print('✅ Fișierul încărcat cu succes!')

print(f'\n📊 Dataset: {len(df)} rânduri, {len(df.columns)} coloane')
print(f'📋 Coloane: {list(df.columns)}')

## Pasul 5 — EDA (Explorarea și Înțelegerea Datelor)

Înainte de orice model, trebuie să înțelegem ce avem. La fel ca la Titanic — am analizat distribuția supraviețuitorilor înainte să construim modelul.

In [ ]:
print('=== PRIMELE 5 RÂNDURI ===')
df.head()

In [ ]:
print('=== INFORMAȚII GENERALE ===')
df.info()
print()
print('=== VALORI LIPSĂ ===')
print(df.isnull().sum())

In [ ]:
print('=== STATISTICI DESCRIPTIVE ===')
df.describe(include='all')

In [ ]:
# GRAFIC 1 — Distribuția tipurilor de agenți patogeni
# Similar cu analiza supraviețuitorilor de la Titanic — vedem distribuția claselor
plt.figure(figsize=(13, 5))
tipuri = df['pathogen_type'].value_counts().head(10)
colors = plt.cm.Set2(np.linspace(0, 1, len(tipuri)))
bars = plt.bar(tipuri.index, tipuri.values, color=colors, edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, tipuri.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(val), ha='center', va='bottom', fontweight='bold')
plt.title('Distribuția tipurilor de agenți patogeni din catalog', fontsize=14, fontweight='bold')
plt.xlabel('Tipul agentului patogen')
plt.ylabel('Număr de înregistrări')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig('results/grafic1_tipuri.png', dpi=150)
plt.show()
print('💡 Observație: Bacteriile și paraziții domină catalogul.')

In [ ]:
# GRAFIC 2 — Distribuția numărului de boli per agent patogen
# Echivalentul histogramei distribuției vârstelor de la Titanic
df['disease_count_num'] = pd.to_numeric(df['disease_count'], errors='coerce')

plt.figure(figsize=(11, 5))
plt.hist(df['disease_count_num'].dropna(), bins=20, color='steelblue',
         edgecolor='white', linewidth=0.8)
media = df['disease_count_num'].dropna().mean()
mediana = df['disease_count_num'].dropna().median()
plt.axvline(media, color='red', linestyle='--', linewidth=2, label=f'Medie: {media:.1f}')
plt.axvline(mediana, color='orange', linestyle='--', linewidth=2, label=f'Mediană: {mediana:.1f}')
plt.title('Câte boli are fiecare agent patogen?', fontsize=14, fontweight='bold')
plt.xlabel('Număr de boli asociate')
plt.ylabel('Număr de agenți patogeni')
plt.legend()
plt.tight_layout()
plt.savefig('results/grafic2_distributie_boli.png', dpi=150)
plt.show()
print(f'💡 Observație: Media = {media:.1f} boli/agent. Distribuția e skewed — unii agenți au zeci de boli.')

In [ ]:
# GRAFIC 3 — Top 15 agenți patogeni cu cele mai multe boli asociate
plt.figure(figsize=(14, 7))
top_agenti = df.nlargest(15, 'disease_count_num')[['pathogen_name', 'disease_count_num', 'pathogen_type']]
top_agenti['name_short'] = top_agenti['pathogen_name'].str[:35]

color_map = {'Bacterie': 'steelblue', 'Parazit': 'orange', 'Necunoscut': 'gray'}
bar_colors = [color_map.get(t, 'green') for t in top_agenti['pathogen_type']]

plt.barh(top_agenti['name_short'], top_agenti['disease_count_num'],
         color=bar_colors, edgecolor='black', linewidth=0.5)
patches = [mpatches.Patch(color=c, label=l) for l, c in color_map.items()]
plt.legend(handles=patches, loc='lower right')
plt.title('Top 15 agenți patogeni cu cele mai multe boli asociate', fontsize=13, fontweight='bold')
plt.xlabel('Număr de boli asociate')
plt.tight_layout()
plt.savefig('results/grafic3_top_agenti.png', dpi=150)
plt.show()
print('💡 Observație: Unii agenți patogeni sunt asociați cu 40-50 de boli diferite!')

In [ ]:
# GRAFIC 4 — Top 20 organe cel mai des afectate
toate_organele = []
for organe_str in df['target_organs'].dropna():
    for organ in str(organe_str).split(';'):
        organ = organ.strip()
        if organ and organ != 'nan':
            toate_organele.append(organ)

freq_organe = Counter(toate_organele)
top_organe = dict(sorted(freq_organe.items(), key=lambda x: x[1], reverse=True)[:20])

plt.figure(figsize=(13, 8))
colors_org = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(top_organe)))
plt.barh(list(top_organe.keys()), list(top_organe.values()),
         color=colors_org, edgecolor='black', linewidth=0.5)
plt.title('Top 20 organe cel mai frecvent afectate de agenții patogeni', fontsize=13, fontweight='bold')
plt.xlabel('Număr de agenți patogeni care afectează organul')
plt.tight_layout()
plt.savefig('results/grafic4_organe.png', dpi=150)
plt.show()
print('💡 Observație: Sângele (blood) și ficatul (hepar) sunt cele mai vulnerabile organe.')

In [ ]:
# GRAFIC 5 — WordCloud din toate bolile asociate
toate_bolile_text = ' '.join(df['associated_diseases'].dropna().tolist())
toate_bolile_text = re.sub(r'[^\w\s]', ' ', toate_bolile_text.lower())

wc = WordCloud(width=1000, height=500, background_color='white',
               colormap='plasma', max_words=100,
               stopwords={'si', 'sau', 'de', 'la', 'in', 'cu', 'pe', 'a', 'o'}).generate(toate_bolile_text)

plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Cele mai frecvente boli și afecțiuni din catalog', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/grafic5_wordcloud.png', dpi=150)
plt.show()
print('💡 Observație: Infecțiile, ACNEE, cancerul și sindroamele complexe apar cel mai des.')

In [ ]:
# GRAFIC 6 — Pie chart: proporția tipurilor de agenți patogeni
tip_counts = df['pathogen_type'].value_counts()
tip_grouped = {}
altele = 0
for tip, count in tip_counts.items():
    if tip in ['Bacterie', 'Parazit', 'Necunoscut']:
        tip_grouped[tip] = count
    else:
        altele += count
if altele > 0:
    tip_grouped['Coduri/Altele'] = altele

plt.figure(figsize=(9, 9))
wedge_props = {'linewidth': 2, 'edgecolor': 'white'}
plt.pie(tip_grouped.values(), labels=tip_grouped.keys(),
        autopct='%1.1f%%', startangle=90,
        colors=['#4C72B0', '#DD8452', '#55A868', '#C44E52'],
        wedgeprops=wedge_props, textprops={'fontsize': 12})
plt.title('Proporția tipurilor de agenți patogeni în catalog', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/grafic6_pie.png', dpi=150)
plt.show()

## Pasul 6 — Preprocesare Text

In [ ]:
def curata_text(text):
    """Curăță textul pentru procesare NLP."""
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'[^\w\săîâșțĂÎÂȘȚ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Completăm valorile lipsă
df['pathogen_type'] = df['pathogen_type'].fillna('Necunoscut')
df['frequency_hz'] = df['frequency_hz'].fillna('Necunoscut')
df['associated_diseases'] = df['associated_diseases'].fillna('')
df['target_organs'] = df['target_organs'].fillna('')

# Creăm versiunile curate
df['boli_curate'] = df['associated_diseases'].apply(curata_text)
df['disease_count_num'] = pd.to_numeric(df['disease_count'], errors='coerce').fillna(0).astype(int)

# Versiuni lowercase pentru căutare
df['boli_lower'] = df['associated_diseases'].str.lower().fillna('')

print('✅ Preprocesarea este gata!')
print(f'\nExemplu text curat: {df["boli_curate"].iloc[5][:80]}...')

## Pasul 7 — Construim Dataset pentru Clasificare Binară

**Ideea cheie:** Transformăm problema de căutare într-o problemă de clasificare binară.
La fel ca la Titanic (supraviețuit = 1 sau 0), noi vom prezice:
- **1** = agentul patogen este relevant pentru boala căutată
- **0** = agentul patogen nu este relevant

Vom antrena modele (XGBoost, SVM, Random Forest) și le vom evalua cu ROC AUC.

In [ ]:
# Construim un dataset de training pentru clasificare binară
# Alegem câteva boli de referință ca să creăm labeluri
BOLI_REFERINTA = [
    'acnee', 'sifilis', 'toxoplasmoza', 'tenie', 'tetanos',
    'cancer', 'leucemie', 'dermatita', 'tuberculoza', 'malarie'
]

rows_ml = []
for boala in BOLI_REFERINTA:
    for idx, rand in df.iterrows():
        # label 1 = agentul patogen e asociat cu boala
        # label 0 = nu e asociat
        label = 1 if boala in rand['boli_lower'] else 0
        rows_ml.append({
            'boala_query': boala,
            'pathogen_text': rand['boli_curate'],
            'disease_count': rand['disease_count_num'],
            'is_bacterie': 1 if rand['pathogen_type'] == 'Bacterie' else 0,
            'is_parazit': 1 if rand['pathogen_type'] == 'Parazit' else 0,
            'text_length': len(str(rand['boli_curate'])),
            'label': label
        })

df_ml = pd.DataFrame(rows_ml)

print(f'Dataset ML creat: {len(df_ml)} rânduri')
print(f'Pozitive (relevante): {df_ml["label"].sum()} ({df_ml["label"].mean():.1%})')
print(f'Negative (nerelevante): {(df_ml["label"]==0).sum()} ({(df_ml["label"]==0).mean():.1%})')
print('\nLa fel ca la Titanic — avem clase dezechilibrate!')
print('Vom folosi ROC AUC ca metrică principală, nu doar accuracy.')

In [ ]:
# Feature Engineering cu TF-IDF
# Transformăm textul în features numerice
print('Se construiesc feature-urile TF-IDF...')

tfidf = TfidfVectorizer(max_features=200, ngram_range=(1,2))

# Combinăm query-ul cu textul agentului pentru TF-IDF
texte_combinate = df_ml['boala_query'] + ' ' + df_ml['pathogen_text']
X_tfidf = tfidf.fit_transform(texte_combinate).toarray()

# Features numerice suplimentare
X_numeric = df_ml[['disease_count', 'is_bacterie', 'is_parazit', 'text_length']].values

# Combinăm TF-IDF cu features numerice
X = np.hstack([X_tfidf, X_numeric])
y = df_ml['label'].values

# Split train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'✅ Features: {X.shape[1]} variabile')
print(f'Train: {len(X_train)} exemple')
print(f'Test:  {len(X_test)} exemple')

# Salvăm vectorizatorul
with open('models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

## Pasul 8 — Antrenarea Modelelor ML

Antrenăm 3 modele și le comparăm:
1. **XGBoost** — boosting pipeline (discutat la curs, câștigă Kaggle)
2. **SVM** — hyperplane optimal separator (Support Vector Machine din curs)
3. **Random Forest** — ansamblu de arbori (baseline solid)

In [ ]:
# ============================================================
# MODEL 1: XGBoost
# Pipeline de clasificatori slabi care se corectează reciproc
# Discutat la curs: câștigă competiții Kaggle
# ============================================================
print('Antrenăm XGBoost...')
xgb = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

print(f'XGBoost Accuracy: {accuracy_score(y_test, y_pred_xgb):.3f}')
print(f'XGBoost ROC AUC:  {roc_auc_score(y_test, y_prob_xgb):.3f}')
print(f'XGBoost F1 Score: {f1_score(y_test, y_pred_xgb):.3f}')

with open('models/model_xgboost.pkl', 'wb') as f:
    pickle.dump(xgb, f)
print('✅ XGBoost salvat!')

In [ ]:
# ============================================================
# MODEL 2: SVM (Support Vector Machine)
# Discutat la curs — găsește hyperplane-ul optim între clase
# ============================================================
print('Antrenăm SVM...')
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm = SVC(kernel='rbf', probability=True, random_state=42, C=1.0)
svm.fit(X_train_scaled, y_train)
y_pred_svm = svm.predict(X_test_scaled)
y_prob_svm = svm.predict_proba(X_test_scaled)[:, 1]

print(f'SVM Accuracy: {accuracy_score(y_test, y_pred_svm):.3f}')
print(f'SVM ROC AUC:  {roc_auc_score(y_test, y_prob_svm):.3f}')
print(f'SVM F1 Score: {f1_score(y_test, y_pred_svm):.3f}')

with open('models/model_svm.pkl', 'wb') as f:
    pickle.dump(svm, f)
print('✅ SVM salvat!')

In [ ]:
# ============================================================
# MODEL 3: Random Forest
# Ansamblu de arbori — baseline solid pentru comparație
# ============================================================
print('Antrenăm Random Forest...')
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print(f'Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}')
print(f'Random Forest ROC AUC:  {roc_auc_score(y_test, y_prob_rf):.3f}')
print(f'Random Forest F1 Score: {f1_score(y_test, y_pred_rf):.3f}')

with open('models/model_rf.pkl', 'wb') as f:
    pickle.dump(rf, f)
print('✅ Random Forest salvat!')

## Pasul 9 — Evaluare și Comparație Modele

Folosim **ROC AUC** ca metrică principală — la fel ca în cursul despre Titanic Kaggle.
ROC AUC e mai bun decât accuracy când avem clase dezechilibrate.

In [ ]:
# GRAFIC 7 — ROC Curves pentru toate cele 3 modele
# Exact cum s-a discutat la curs despre ROC AUC
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Stânga: ROC Curves ---
ax = axes[0]
modele_info = [
    ('XGBoost', y_prob_xgb, '#2196F3'),
    ('SVM', y_prob_svm, '#FF9800'),
    ('Random Forest', y_prob_rf, '#4CAF50'),
]

for name, y_prob, color in modele_info:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, color=color, linewidth=2.5, label=f'{name} (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.500)')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Comparație Modele\n(Discutată la curs — Titanic Kaggle)', fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# --- Dreapta: Tabel comparativ cu toate metricile ---
ax2 = axes[1]
ax2.axis('off')

date_tabel = []
for name, y_pred, y_prob in [
    ('XGBoost', y_pred_xgb, y_prob_xgb),
    ('SVM', y_pred_svm, y_prob_svm),
    ('Random Forest', y_pred_rf, y_prob_rf)
]:
    date_tabel.append([
        name,
        f'{accuracy_score(y_test, y_pred):.3f}',
        f'{precision_score(y_test, y_pred, zero_division=0):.3f}',
        f'{recall_score(y_test, y_pred, zero_division=0):.3f}',
        f'{f1_score(y_test, y_pred, zero_division=0):.3f}',
        f'{roc_auc_score(y_test, y_prob):.3f}'
    ])

col_labels = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC AUC']
tabel = ax2.table(cellText=date_tabel, colLabels=col_labels,
                  loc='center', cellLoc='center')
tabel.auto_set_font_size(False)
tabel.set_fontsize(11)
tabel.scale(1.2, 2.2)

for (r, c), cell in tabel.get_celld().items():
    if r == 0:
        cell.set_facecolor('#37474F')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#F5F5F5')

ax2.set_title('Tabel Comparativ Metrici', fontsize=12, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('results/grafic7_roc_comparatie.png', dpi=150)
plt.show()
print('💡 ROC AUC > 0.5 înseamnă că modelul e mai bun decât ghicitul aleator.')
print('💡 Cu cât AUC e mai aproape de 1.0, cu atât modelul e mai bun.')

In [ ]:
# GRAFIC 8 — Confusion Matrix pentru XGBoost (cel mai bun model)
# La fel ca la analiza supraviețuitorilor Titanic
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Nerelevant (0)', 'Relevant (1)'],
            yticklabels=['Nerelevant (0)', 'Relevant (1)'])
axes[0].set_title('Confusion Matrix — XGBoost', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Valoare Reală')
axes[0].set_xlabel('Valoare Prezisă')

# Feature Importance XGBoost
importances = xgb.feature_importances_
top_idx = np.argsort(importances)[::-1][:15]

feature_names = tfidf.get_feature_names_out().tolist() + ['disease_count', 'is_bacterie', 'is_parazit', 'text_length']
top_features = [feature_names[i] if i < len(feature_names) else f'feat_{i}' for i in top_idx]
top_importances = importances[top_idx]

axes[1].barh(top_features[::-1], top_importances[::-1], color='steelblue', edgecolor='black', linewidth=0.5)
axes[1].set_title('Top 15 Feature Importances — XGBoost', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importanță')

plt.tight_layout()
plt.savefig('results/grafic8_confusion_features.png', dpi=150)
plt.show()
print('💡 Feature Importance: ce variabile contează cel mai mult pentru XGBoost?')

In [ ]:
# Cross Validation — evaluare mai robustă
print('=== CROSS VALIDATION (5-fold) ===')
print('Mai robust decât un singur split train/test\n')

cv_scores_xgb = cross_val_score(xgb, X, y, cv=5, scoring='roc_auc')
cv_scores_rf = cross_val_score(rf, X, y, cv=5, scoring='roc_auc')

print(f'XGBoost   ROC AUC: {cv_scores_xgb.mean():.3f} ± {cv_scores_xgb.std():.3f}')
print(f'RandomForest ROC AUC: {cv_scores_rf.mean():.3f} ± {cv_scores_rf.std():.3f}')

print()
print('=== CLASSIFICATION REPORT — XGBoost ===')
print(classification_report(y_test, y_pred_xgb,
      target_names=['Nerelevant (0)', 'Relevant (1)']))

## Pasul 10 — Modelul Semantic AI (Sentence Transformers)

Pe lângă modelele ML clasice, adăugăm și un model AI care înțelege **sensul** textului.

In [ ]:
print('Se încarcă modelul Sentence Transformer... (1-2 minute prima dată)')
model_semantic = SentenceTransformer('all-MiniLM-L6-v2')
print('✅ Model semantic încărcat!')

print('Se calculează embeddingurile pentru toți agenții patogeni...')
embeddings = model_semantic.encode(df['boli_curate'].tolist(), show_progress_bar=True)

np.save('models/embeddings_semantic.npy', embeddings)
print(f'✅ Embeddings salvate! Shape: {embeddings.shape}')

## Pasul 11 — Sistemul Final de Căutare cu Chat Interface

Acum combinăm totul: căutare directă → XGBoost → Semantic AI

In [ ]:
def construieste_agent_pydantic(rand, scor=0.9):
    """Construiește un obiect AgentPatogen validat cu Pydantic."""
    boli = [b.strip() for b in str(rand['associated_diseases']).split(';') if b.strip()]
    organe = [o.strip() for o in str(rand['target_organs']).split(';') if o.strip()]
    return AgentPatogen(
        nume=str(rand['pathogen_name']),
        tip=str(rand['pathogen_type']),
        frecventa=str(rand['frequency_hz']),
        boli_asociate=boli[:5],
        organe_tinta=organe[:5],
        numar_boli=int(rand['disease_count_num']),
        scor_relevanta=round(min(max(float(scor), 0.0), 1.0), 3)
    )


def cauta_direct(boala, top_n=5):
    """Model 1: Căutare directă — rapid și precis pentru termeni exacți."""
    mask = df['boli_lower'].str.contains(boala.lower().strip(), na=False, regex=False)
    rezultate = df[mask].head(top_n)
    agenti = [construieste_agent_pydantic(r, 0.95) for _, r in rezultate.iterrows()]
    return agenti, 'Cautare Directa'


def cauta_semantic(boala, top_n=5):
    """Model 2: Semantic AI — înțelege sensul textului."""
    emb_query = model_semantic.encode([boala.lower()])
    scoruri = cosine_similarity(emb_query, embeddings)[0]
    idx_top = scoruri.argsort()[::-1][:top_n*3]
    idx_relevanti = [i for i in idx_top if scoruri[i] > 0.15][:top_n]
    agenti = [construieste_agent_pydantic(df.iloc[i], scoruri[i]) for i in idx_relevanti]
    return agenti, 'Semantic AI'


def gaseste_agenti_patogeni(boala_cautata, top_n=5, verbose=True):
    """
    Funcția principală a sistemului.
    Strategie: Direct → Semantic
    Returnează RezultatCautare validat cu Pydantic.
    """
    agenti, model_name = cauta_direct(boala_cautata, top_n)

    if len(agenti) < 2:
        if verbose:
            print('   → Căutare directă insuficientă, trec la Semantic AI...')
        agenti, model_name = cauta_semantic(boala_cautata, top_n)

    rezultat = RezultatCautare(
        boala_cautata=boala_cautata,
        agenti_gasiti=agenti,
        numar_rezultate=len(agenti),
        model_folosit=model_name,
        mesaj=f'[{model_name}] Am găsit {len(agenti)} agenți patogeni pentru "{boala_cautata}"'
    )
    return rezultat


def afiseaza_rezultat(rezultat):
    """Afișează rezultatul într-un format clar și ușor de citit."""
    print(f'\n🔍 {rezultat.mesaj}')
    print('━' * 60)

    if rezultat.numar_rezultate == 0:
        print('⚠️  Nu am găsit niciun agent patogen.')
        print('   Încearcă alt cuvânt cheie sau verifică ortografia.')
        return

    for i, agent in enumerate(rezultat.agenti_gasiti, 1):
        bara = '█' * int(agent.scor_relevanta * 20)
        print(f'\n  {i}. 🦠 {agent.nume}')
        print(f'     Tip:        {agent.tip}')
        print(f'     Frecvență:  {agent.frecventa} Hz')
        print(f'     Relevanță:  {bara} {agent.scor_relevanta:.0%}')
        print(f'     Nr. boli:   {agent.numar_boli}')
        if agent.boli_asociate:
            print(f'     Boli:       {" | ".join(agent.boli_asociate[:3])}')
        if agent.organe_tinta:
            print(f'     Organe:     {" | ".join(agent.organe_tinta[:3])}')

    print(f'\n  📌 Model folosit: {rezultat.model_folosit}')


print('✅ Sistemul final este gata!')

## Pasul 12 — 🎯 INTERFAȚA DE CHAT

**Scrie boala ta mai jos și rulează celula cu Shift+Enter!**

In [ ]:
# ================================================================
#  🎯 SCRIE BOALA TA AICI și apasă Shift+Enter
# ================================================================
#
#  Exemple: 'ACNEE', 'sifilis', 'toxoplasmoza', 'tenie',
#            'tetanos', 'boala Lyme', 'TUSE', 'leucemie'
#
BOALA_CAUTATA = 'ACNEE'   # <-- SCHIMBA AICI
#
# ================================================================

rezultat = gaseste_agenti_patogeni(BOALA_CAUTATA, top_n=5)
afiseaza_rezultat(rezultat)

print()
print('=== DATE JSON (structură Pydantic validată) ===')
import json
data_json = json.loads(rezultat.model_dump_json())
data_json['agenti_gasiti'] = data_json['agenti_gasiti'][:2]  # primii 2 pentru preview
print(json.dumps(data_json, indent=2, ensure_ascii=False)[:800], '...')

In [ ]:
# Test pe mai multe boli
boli_demo = ['sifilis', 'toxoplasmoza', 'tetanos', 'boala Lyme']

for boala in boli_demo:
    rez = gaseste_agenti_patogeni(boala, top_n=3, verbose=False)
    afiseaza_rezultat(rez)
    print('─' * 60)

## Pasul 13 — Salvare date și modele

In [ ]:
df.to_csv('data/patogeni_procesati.csv', index=False)

print('✅ Fișiere salvate:')
print('  📁 data/patogeni_procesati.csv')
print('  📁 models/model_xgboost.pkl')
print('  📁 models/model_svm.pkl')
print('  📁 models/model_rf.pkl')
print('  📁 models/tfidf_vectorizer.pkl')
print('  📁 models/embeddings_semantic.npy')
print()
print('  📊 results/grafic1_tipuri.png')
print('  📊 results/grafic2_distributie_boli.png')
print('  📊 results/grafic3_top_agenti.png')
print('  📊 results/grafic4_organe.png')
print('  📊 results/grafic5_wordcloud.png')
print('  📊 results/grafic6_pie.png')
print('  📊 results/grafic7_roc_comparatie.png')
print('  📊 results/grafic8_confusion_features.png')

## 📌 Concluzii

### Ce am realizat
| Element | Detalii |
|---------|----------|
| **Dataset** | 700+ agenți patogeni din vega_pathogens_catalog.csv |
| **EDA** | 8 grafice: distribuții, top agenți, organe, wordcloud, pie |
| **Pydantic** | Validare automată a tuturor rezultatelor returnate |
| **Modele ML** | XGBoost, SVM, Random Forest — comparate cu ROC AUC |
| **Model AI** | Sentence Transformers (BERT-based) pentru căutare semantică |
| **Evaluare** | ROC Curves, Confusion Matrix, Cross-Validation, F1 Score |
| **Sistem final** | Chat interface cu fallback automat între modele |

### Legăturile cu cursul
- **XGBoost** — pipeline de clasificatori slabi care se corectează reciproc (discutat la curs)
- **SVM** — Support Vector Machine cu kernel RBF (discutat la curs)
- **ROC AUC** — metrica principală pentru clase dezechilibrate (Titanic Kaggle din curs)
- **Naive Bayes** — am folosit conceptul de clasificator baseline înainte de XGBoost
- **TF-IDF** — vectorizare text (discutat în lecțiile de NLP)
- **Pydantic** — validare date structurate

### Limitări
- Modelul semantic e antrenat pe engleză, nu pe română
- Dataset relativ mic pentru antrenarea ML (700 intrări)
- **Nu este o aplicație medicală** — nu înlocuiește un medic!

### Îmbunătățiri viitoare
- [ ] Interfață Gradio/Streamlit
- [ ] API REST cu FastAPI
- [ ] Model semantic antrenat pe date medicale românești
- [ ] Căutare și după organe țintă